# Local Qwen3 Analytics RAG

Notebook ini membaca package v2 dari `notebooks/output/latest.json`, menyiapkan semantic index TF-IDF, lalu menampilkan widget chat. Resolver deterministik memilih evidence dan area dari package; Qwen3-8B dipanggil satu kali hanya untuk menyusun penjelasan teks. Geometry final selalu diambil dari package analysis dan overlay dirender dengan Matplotlib.

Default: GGUF resmi `Qwen3-8B-Q4_K_M` melalui llama.cpp dengan thinking dinonaktifkan. Model diunduh otomatis ke Application Support dan tidak disimpan di repository.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys

BACKEND_ROOT = next(candidate for candidate in (Path.cwd(), Path.cwd().parent) if (candidate / 'explanatory_analysis').is_dir())
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from explanatory_analysis.local_model import get_model_runtime
from explanatory_analysis.rag import LocalRAG, RAGConfig

get_model_runtime().ensure_ready()
rag_config = RAGConfig.default()
rag = LocalRAG(rag_config)
health = rag.health()
display(pd.DataFrame([{"jobId": rag.package.job_id, "package": str(rag.package.path), "runtime": health.get("runtime"), "chatModel": rag_config.chat_model, "ready": health["ready"]}]))
if not health["ready"]:
    raise RuntimeError(f"Runtime Qwen3-8B belum siap: {health}")
index_status = rag.prepare_index()
display(Markdown(f"Semantic index TF-IDF siap: **{index_status['documentCount']} evidence cards**."))


## Tanya data pantry

Resolver menyimpan audit grounding deterministik, evidence, jawaban, dan floorplan overlay. Qwen3-8B tidak memilih area dan tidak diminta menghasilkan JSON. Run baru disimpan ke `llm-rag-v2/runs/`; riwayat lama tetap dapat dibuka sebagai legacy.


In [ ]:
chat_widget = rag.widget()
display(chat_widget)


Programmatic API tetap tersedia:

```python
result = rag.ask("Area mana yang paling sering dilewati?")
```
